In [2]:
# =========================
# CELL 1: Imports and config
# =========================

import os
import random
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from torchvision import datasets, transforms
import torchvision.transforms.functional as TF

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

DATA_DIR = "dataset location"

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 100
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

USE_CLASS_WEIGHTS = True
LABEL_SMOOTHING = 0.05

BEST_MODEL_PATH = "best_scg_lgt.pth"

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# This means:
# 1 original image + 6 augmentation versions
# Total training expansion = 7x
INCLUDE_ORIGINAL_IN_TRAIN = True

AUGMENTATION_TECHNIQUES = [
    "flipping",
    "gamma_correction",
    "noise_injection",
    "pca_color_augmentation",
    "rotation",
    "scaling"
]

print("Requested augmentation techniques:", AUGMENTATION_TECHNIQUES)

Device: cuda
Requested augmentation techniques: ['flipping', 'gamma_correction', 'noise_injection', 'pca_color_augmentation', 'rotation', 'scaling']


In [ ]:
# =========================
# CELL 2: Dataset loading and explicit six-technique augmentation
# =========================

class PCALighting:
    """
    PCA color augmentation, also known as AlexNet-style lighting noise.

    This applies small RGB perturbations along principal color directions.
    It is useful for making the model less sensitive to illumination/color shifts.
    """

    def __init__(self, alpha_std=0.1):
        self.alpha_std = alpha_std

        self.eigvals = torch.tensor([0.2175, 0.0188, 0.0045], dtype=torch.float32)

        self.eigvecs = torch.tensor([
            [-0.5675,  0.7192,  0.4009],
            [-0.5808, -0.0045, -0.8140],
            [-0.5836, -0.6948,  0.4203]
        ], dtype=torch.float32)

    def __call__(self, img_tensor):
        """
        img_tensor shape: [3, H, W], value range: [0, 1]
        """

        alpha = torch.normal(
            mean=0.0,
            std=self.alpha_std,
            size=(3,)
        )

        rgb_noise = (self.eigvecs * (alpha * self.eigvals).view(1, 3)).sum(dim=1)

        img_tensor = img_tensor + rgb_noise.view(3, 1, 1)
        img_tensor = torch.clamp(img_tensor, 0.0, 1.0)

        return img_tensor


class SixTechniqueAugmentedDataset(Dataset):
    """
    Expands the training dataset using exactly six augmentation techniques:

    1. Image flipping
    2. Gamma correction
    3. Noise injection
    4. PCA color augmentation
    5. Rotation
    6. Scaling

    If INCLUDE_ORIGINAL_IN_TRAIN = True:
        dataset size = original size x 7

    If INCLUDE_ORIGINAL_IN_TRAIN = False:
        dataset size = original size x 6
    """

    def __init__(
        self,
        base_dataset,
        img_size=224,
        mean=MEAN,
        std=STD,
        include_original=True
    ):
        self.base_dataset = base_dataset
        self.img_size = img_size
        self.mean = mean
        self.std = std
        self.include_original = include_original

        self.augmentation_names = [
            "flipping",
            "gamma_correction",
            "noise_injection",
            "pca_color_augmentation",
            "rotation",
            "scaling"
        ]

        if self.include_original:
            self.mode_names = ["original"] + self.augmentation_names
        else:
            self.mode_names = self.augmentation_names

        self.multiplier = len(self.mode_names)

        self.resize_center_to_tensor = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.CenterCrop(self.img_size),
            transforms.ToTensor()
        ])

        self.normalize = transforms.Normalize(
            mean=self.mean,
            std=self.std
        )

        self.rotation_transform = transforms.RandomRotation(
            degrees=20
        )

        self.scaling_transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomResizedCrop(
                self.img_size,
                scale=(0.75, 1.0),
                ratio=(0.9, 1.1)
            ),
            transforms.ToTensor()
        ])

        self.pca_lighting = PCALighting(alpha_std=0.1)

        self.targets = self._get_repeated_targets()

    def _get_base_targets(self):
        if isinstance(self.base_dataset, Subset):
            return [self.base_dataset.dataset.targets[i] for i in self.base_dataset.indices]

        if hasattr(self.base_dataset, "targets"):
            return self.base_dataset.targets

        raise AttributeError("Base dataset must have targets or be a Subset of a dataset with targets.")

    def _get_repeated_targets(self):
        base_targets = self._get_base_targets()
        return base_targets * self.multiplier

    def __len__(self):
        return len(self.base_dataset) * self.multiplier

    def _apply_flipping(self, image):
        flip_choice = random.choice(["horizontal", "vertical", "both"])

        if flip_choice == "horizontal":
            image = TF.hflip(image)

        elif flip_choice == "vertical":
            image = TF.vflip(image)

        else:
            image = TF.hflip(image)
            image = TF.vflip(image)

        return image

    def _apply_gamma_correction(self, image):
        gamma = random.uniform(0.7, 1.5)
        image = TF.adjust_gamma(image, gamma=gamma, gain=1.0)
        return image

    def _apply_noise_injection(self, img_tensor):
        noise_std = random.uniform(0.01, 0.05)
        noise = torch.randn_like(img_tensor) * noise_std
        img_tensor = img_tensor + noise
        img_tensor = torch.clamp(img_tensor, 0.0, 1.0)
        return img_tensor

    def __getitem__(self, idx):
        base_idx = idx % len(self.base_dataset)
        mode_idx = idx // len(self.base_dataset)

        image, label = self.base_dataset[base_idx]

        if not isinstance(image, Image.Image):
            raise TypeError("Expected PIL image. Make sure the base ImageFolder dataset uses transform=None.")

        image = image.convert("RGB")
        mode = self.mode_names[mode_idx]

        if mode == "original":
            img_tensor = self.resize_center_to_tensor(image)

        elif mode == "flipping":
            image = self._apply_flipping(image)
            img_tensor = self.resize_center_to_tensor(image)

        elif mode == "gamma_correction":
            image = self._apply_gamma_correction(image)
            img_tensor = self.resize_center_to_tensor(image)

        elif mode == "noise_injection":
            img_tensor = self.resize_center_to_tensor(image)
            img_tensor = self._apply_noise_injection(img_tensor)

        elif mode == "pca_color_augmentation":
            img_tensor = self.resize_center_to_tensor(image)
            img_tensor = self.pca_lighting(img_tensor)

        elif mode == "rotation":
            image = self.rotation_transform(image)
            img_tensor = self.resize_center_to_tensor(image)

        elif mode == "scaling":
            img_tensor = self.scaling_transform(image)

        else:
            raise ValueError(f"Unknown augmentation mode: {mode}")

        img_tensor = self.normalize(img_tensor)

        return img_tensor, label


eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=MEAN,
        std=STD
    )
])


def find_split_dirs(root):
    names = os.listdir(root)
    lower_map = {name.lower(): name for name in names}

    train_keys = ["train", "training"]
    val_keys = ["val", "valid", "validation"]
    test_keys = ["test", "testing"]

    train_dir = next((os.path.join(root, lower_map[k]) for k in train_keys if k in lower_map), None)
    val_dir = next((os.path.join(root, lower_map[k]) for k in val_keys if k in lower_map), None)
    test_dir = next((os.path.join(root, lower_map[k]) for k in test_keys if k in lower_map), None)

    if train_dir and val_dir and test_dir:
        return train_dir, val_dir, test_dir

    return None, None, None


def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


generator = torch.Generator()
generator.manual_seed(SEED)


train_dir, val_dir, test_dir = find_split_dirs(DATA_DIR)

if train_dir and val_dir and test_dir:
    train_base_dataset = datasets.ImageFolder(
        train_dir,
        transform=None
    )

    val_dataset = datasets.ImageFolder(
        val_dir,
        transform=eval_transform
    )

    test_dataset = datasets.ImageFolder(
        test_dir,
        transform=eval_transform
    )

    class_names = train_base_dataset.classes

    assert train_base_dataset.class_to_idx == val_dataset.class_to_idx
    assert train_base_dataset.class_to_idx == test_dataset.class_to_idx

else:
    full_base_dataset = datasets.ImageFolder(
        DATA_DIR,
        transform=None
    )

    full_eval_dataset = datasets.ImageFolder(
        DATA_DIR,
        transform=eval_transform
    )

    targets = np.array(full_base_dataset.targets)
    indices = np.arange(len(targets))

    try:
        train_idx, temp_idx = train_test_split(
            indices,
            test_size=0.30,
            random_state=SEED,
            stratify=targets
        )

        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=0.50,
            random_state=SEED,
            stratify=targets[temp_idx]
        )

    except ValueError:
        train_idx, temp_idx = train_test_split(
            indices,
            test_size=0.30,
            random_state=SEED,
            shuffle=True
        )

        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=0.50,
            random_state=SEED,
            shuffle=True
        )

    train_base_dataset = Subset(
        full_base_dataset,
        train_idx
    )

    val_dataset = Subset(
        full_eval_dataset,
        val_idx
    )

    test_dataset = Subset(
        full_eval_dataset,
        test_idx
    )

    class_names = full_base_dataset.classes


train_dataset = SixTechniqueAugmentedDataset(
    base_dataset=train_base_dataset,
    img_size=IMG_SIZE,
    mean=MEAN,
    std=STD,
    include_original=INCLUDE_ORIGINAL_IN_TRAIN
)

num_classes = len(class_names)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Classes:", class_names)
print("Number of classes:", num_classes)

print("Original train samples:", len(train_base_dataset))
print("Expanded train samples:", len(train_dataset))
print("Training expansion multiplier:", train_dataset.multiplier)
print("Applied training modes:", train_dataset.mode_names)

print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

In [4]:
# =========================
# CELL 3: Model components
# =========================

class ConvBNAct(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, groups=1, act=True):
        super().__init__()

        padding = kernel_size // 2

        layers = [
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                groups=groups,
                bias=False
            ),
            nn.BatchNorm2d(out_channels)
        ]

        if act:
            layers.append(nn.SiLU(inplace=True))

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class ECALayer(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super().__init__()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        self.conv = nn.Conv1d(
            in_channels=1,
            out_channels=1,
            kernel_size=kernel_size,
            padding=(kernel_size - 1) // 2,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y)
        y = self.sigmoid(y)
        y = y.transpose(-1, -2).unsqueeze(-1)

        return x * y.expand_as(x)


class SharedStem(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            ConvBNAct(3, 24, kernel_size=3, stride=2),
            ConvBNAct(24, 24, kernel_size=3, stride=2, groups=24),
            ConvBNAct(24, 48, kernel_size=1, stride=1)
        )

    def forward(self, x):
        return self.stem(x)


class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, expansion=4, use_eca=True):
        super().__init__()

        hidden_dim = in_channels * expansion
        self.use_residual = stride == 1 and in_channels == out_channels

        layers = []

        if expansion != 1:
            layers.append(
                ConvBNAct(
                    in_channels,
                    hidden_dim,
                    kernel_size=1
                )
            )

        layers.append(
            ConvBNAct(
                hidden_dim,
                hidden_dim,
                kernel_size=3,
                stride=stride,
                groups=hidden_dim
            )
        )

        if use_eca:
            layers.append(
                ECALayer(hidden_dim)
            )

        layers.append(
            ConvBNAct(
                hidden_dim,
                out_channels,
                kernel_size=1,
                act=False
            )
        )

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_residual:
            return x + self.block(x)

        return self.block(x)


class CNNLocalBranch(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            MBConv(48, 64, stride=2, expansion=4),
            MBConv(64, 96, stride=2, expansion=4),
            ECALayer(96)
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)

        return x


class TransformerGlobalBranch(nn.Module):
    def __init__(
        self,
        in_channels=48,
        feature_map_size=56,
        patch_size=4,
        embed_dim=128,
        depth=2,
        num_heads=2,
        mlp_ratio=2,
        dropout=0.1
    ):
        super().__init__()

        self.patch_size = patch_size
        self.num_patches = (feature_map_size // patch_size) ** 2

        self.patch_embed = nn.Conv2d(
            in_channels=in_channels,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

        self.pos_embed = nn.Parameter(
            torch.zeros(1, self.num_patches, embed_dim)
        )

        self.pos_drop = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * mlp_ratio,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=depth
        )

        self.norm = nn.LayerNorm(embed_dim)

        nn.init.trunc_normal_(
            self.pos_embed,
            std=0.02
        )

    def forward(self, x):
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)

        x = x + self.pos_embed
        x = self.pos_drop(x)

        x = self.encoder(x)
        x = self.norm(x)

        x = x.mean(dim=1)

        return x


class CrossGatedFusion(nn.Module):
    def __init__(self, dim=256, hidden_dim=64):
        super().__init__()

        self.gate_cnn = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim),
            nn.Sigmoid()
        )

        self.gate_tr = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim),
            nn.Sigmoid()
        )

        self.norm = nn.LayerNorm(dim)

    def forward(self, f_cnn, f_tr):
        g_cnn = self.gate_cnn(f_tr)
        g_tr = self.gate_tr(f_cnn)

        f_cnn_refined = g_cnn * f_cnn
        f_tr_refined = g_tr * f_tr

        fused = f_cnn_refined + f_tr_refined
        fused = self.norm(fused)

        return fused, g_cnn, g_tr


class CGLGCTNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.shared_stem = SharedStem()

        self.cnn_branch = CNNLocalBranch()

        self.transformer_branch = TransformerGlobalBranch(
            in_channels=48,
            feature_map_size=56,
            patch_size=4,
            embed_dim=128,
            depth=2,
            num_heads=2,
            mlp_ratio=2,
            dropout=0.1
        )

        self.cnn_proj = nn.Sequential(
            nn.Linear(96, 256),
            nn.LayerNorm(256)
        )

        self.tr_proj = nn.Sequential(
            nn.Linear(128, 256),
            nn.LayerNorm(256)
        )

        self.fusion = CrossGatedFusion(
            dim=256,
            hidden_dim=64
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(256),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        shared_features = self.shared_stem(x)

        f_cnn = self.cnn_branch(shared_features)
        f_tr = self.transformer_branch(shared_features)

        f_cnn = self.cnn_proj(f_cnn)
        f_tr = self.tr_proj(f_tr)

        fused, g_cnn, g_tr = self.fusion(f_cnn, f_tr)

        logits = self.classifier(fused)

        return logits, g_cnn, g_tr

In [5]:
# =========================
# CELL 4: Initialize model
# =========================

model = CGLGCTNet(
    num_classes=num_classes
).to(DEVICE)

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

C:\Users\ASUS\anaconda3\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Total parameters: 595407
Trainable parameters: 595407


In [6]:
# =========================
# CELL 5: Loss, optimizer, scheduler
# =========================

def get_dataset_targets(dataset):
    if hasattr(dataset, "targets"):
        return dataset.targets

    if isinstance(dataset, Subset):
        return [dataset.dataset.targets[i] for i in dataset.indices]

    raise AttributeError("Dataset does not expose targets.")


if USE_CLASS_WEIGHTS:
    train_targets = np.array(
        get_dataset_targets(train_dataset)
    )

    class_counts = np.bincount(
        train_targets,
        minlength=num_classes
    )

    class_weights = len(train_targets) / (
        num_classes * np.maximum(class_counts, 1)
    )

    class_weights = torch.tensor(
        class_weights,
        dtype=torch.float32
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=LABEL_SMOOTHING
    )

    print("Expanded class counts:", class_counts)
    print("Class weights:", class_weights.detach().cpu().numpy())

else:
    criterion = nn.CrossEntropyLoss(
        label_smoothing=LABEL_SMOOTHING
    )


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

try:
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(DEVICE.type == "cuda")
    )
except TypeError:
    scaler = torch.cuda.amp.GradScaler(
        enabled=(DEVICE.type == "cuda")
    )

Expanded class counts: [ 4900  4900  4900  8064  7357  5152  4900  4900  5838  4900  5691  5782
  6776  5271  4900 26985 11256  4900  4900  7245  4900  4900  4900  4900
 24941  8995  5432  4900 10423  4900  9352  4900  8680  8211  6881 26250
  4900  7798]
Class weights: [1.5879699  1.5879699  1.5879699  0.9649123  1.0576394  1.5102975
 1.5879699  1.5879699  1.3328285  1.5879699  1.3672558  1.3457372
 1.1483253  1.4762005  1.5879699  0.28834733 0.6912804  1.5879699
 1.5879699  1.0739893  1.5879699  1.5879699  1.5879699  1.5879699
 0.31197837 0.865042   1.4324471  1.5879699  0.74652714 1.5879699
 0.83202016 1.5879699  0.89643466 0.9476376  1.1308026  0.29642105
 1.5879699  0.9978267 ]


In [7]:
# =========================
# CELL 6: Training and evaluation functions
# =========================

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda")
        ):
            logits, g_cnn, g_tr = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(
            logits,
            dim=1
        )

        all_preds.extend(
            preds.detach().cpu().numpy()
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(
        all_labels,
        all_preds
    )

    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []
    all_g_cnn = []
    all_g_tr = []

    for images, labels in loader:
        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type=device.type,
            enabled=(device.type == "cuda")
        ):
            logits, g_cnn, g_tr = model(images)
            loss = criterion(logits, labels)

        probs = torch.softmax(
            logits,
            dim=1
        )

        preds = torch.argmax(
            logits,
            dim=1
        )

        running_loss += loss.item() * images.size(0)

        all_preds.extend(
            preds.detach().cpu().numpy()
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

        all_probs.extend(
            probs.detach().cpu().numpy()
        )

        all_g_cnn.extend(
            g_cnn.detach().cpu().numpy()
        )

        all_g_tr.extend(
            g_tr.detach().cpu().numpy()
        )

    epoch_loss = running_loss / len(loader.dataset)

    epoch_acc = accuracy_score(
        all_labels,
        all_preds
    )

    return {
        "loss": epoch_loss,
        "accuracy": epoch_acc,
        "labels": np.array(all_labels),
        "preds": np.array(all_preds),
        "probs": np.array(all_probs),
        "g_cnn": np.array(all_g_cnn),
        "g_tr": np.array(all_g_tr)
    }

In [7]:
'''from pathlib import Path
from PIL import Image, UnidentifiedImageError

dataset_root = Path("CCMT Dataset 3")

bad_images = []

valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for image_path in dataset_root.rglob("*"):
    if image_path.suffix.lower() in valid_extensions:
        try:
            with Image.open(image_path) as img:
                img.verify()
        except (UnidentifiedImageError, OSError, ValueError) as e:
            bad_images.append((image_path, str(e)))

print(f"Bad images found: {len(bad_images)}")

for path, error in bad_images:
    print(path, "=>", error)'''

'from pathlib import Path\nfrom PIL import Image, UnidentifiedImageError\n\ndataset_root = Path("CCMT Dataset 3")\n\nbad_images = []\n\nvalid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}\n\nfor image_path in dataset_root.rglob("*"):\n    if image_path.suffix.lower() in valid_extensions:\n        try:\n            with Image.open(image_path) as img:\n                img.verify()\n        except (UnidentifiedImageError, OSError, ValueError) as e:\n            bad_images.append((image_path, str(e)))\n\nprint(f"Bad images found: {len(bad_images)}")\n\nfor path, error in bad_images:\n    print(path, "=>", error)'

In [ ]:
# =========================
# CELL 7: Train model
# =========================

best_val_f1 = -1.0

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_macro_f1": []
}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=DEVICE
    )

    val_results = evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE
    )

    scheduler.step()

    val_labels = val_results["labels"]
    val_preds = val_results["preds"]

    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        val_labels,
        val_preds,
        average="macro",
        zero_division=0
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_results["loss"])
    history["val_acc"].append(val_results["accuracy"])
    history["val_macro_f1"].append(macro_f1)

    if macro_f1 > best_val_f1:
        best_val_f1 = macro_f1

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "class_names": class_names,
                "num_classes": num_classes,
                "epoch": epoch,
                "best_val_f1": best_val_f1,
                "augmentation_modes": train_dataset.mode_names
            },
            BEST_MODEL_PATH
        )

    print(
        f"Epoch [{epoch:03d}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Val Loss: {val_results['loss']:.4f} "
        f"Val Acc: {val_results['accuracy']:.4f} "
        f"Val Macro F1: {macro_f1:.4f}"
    )

print("Best validation macro F1:", best_val_f1)
print("Best model saved to:", BEST_MODEL_PATH)

Epoch [001/100] Train Loss: 0.9958 Train Acc: 0.8438 Val Loss: 0.6138 Val Acc: 0.9625 Val Macro F1: 0.9588


In [ ]:
# =========================
# CELL 8: Plot training history
# =========================

plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history["train_acc"], label="Train Accuracy")
plt.plot(history["val_acc"], label="Validation Accuracy")
plt.plot(history["val_macro_f1"], label="Validation Macro F1")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Accuracy and Macro F1")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# =========================
# CELL 9: Test evaluation
# =========================

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(DEVICE)

test_results = evaluate(
    model=model,
    loader=test_loader,
    criterion=criterion,
    device=DEVICE
)

y_true = test_results["labels"]
y_pred = test_results["preds"]

test_acc = accuracy_score(
    y_true,
    y_pred
)

macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Test Accuracy:", test_acc)
print("Macro Precision:", macro_precision)
print("Macro Recall:", macro_recall)
print("Macro F1:", macro_f1)
print("Weighted Precision:", weighted_precision)
print("Weighted Recall:", weighted_recall)
print("Weighted F1:", weighted_f1)

print("\nClassification Report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        zero_division=0
    )
)

In [ ]:
# =========================
# CELL 10: Confusion matrix
# =========================

cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(8, 6))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.xticks(
    np.arange(num_classes),
    class_names,
    rotation=90
)

plt.yticks(
    np.arange(num_classes),
    class_names
)

plt.colorbar()

for i in range(num_classes):
    for j in range(num_classes):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center",
            color="black"
        )

plt.tight_layout()
plt.show()


class_totals = cm.sum(axis=1)

class_accuracy = np.divide(
    cm.diagonal(),
    class_totals,
    out=np.zeros_like(cm.diagonal(), dtype=float),
    where=class_totals != 0
)

for name, acc in zip(class_names, class_accuracy):
    print(f"{name}: {acc:.4f}")

In [ ]:
class_accuracy = cm.diagonal() / cm.sum(axis=1)

for name, acc in zip(class_names, class_accuracy):
    print(f"{name}: {acc:.4f}")

In [ ]:
# =========================
# CELL 11: Analyze cross-gated fusion
# =========================

g_cnn_values = test_results["g_cnn"]
g_tr_values = test_results["g_tr"]

mean_g_cnn_per_sample = g_cnn_values.mean(axis=1)
mean_g_tr_per_sample = g_tr_values.mean(axis=1)

overall_mean_g_cnn = mean_g_cnn_per_sample.mean()
overall_mean_g_tr = mean_g_tr_per_sample.mean()

print("Overall mean Gcnn:", overall_mean_g_cnn)
print("Overall mean Gtr:", overall_mean_g_tr)

print("\nInterpretation:")
print("Higher Gcnn means stronger transformer-guided CNN feature activation.")
print("Higher Gtr means stronger CNN-guided transformer feature activation.")

print("\nClass-wise gate analysis:")

for class_idx, class_name in enumerate(class_names):
    class_mask = y_true == class_idx

    if class_mask.sum() > 0:
        class_g_cnn = mean_g_cnn_per_sample[class_mask].mean()
        class_g_tr = mean_g_tr_per_sample[class_mask].mean()

        print(
            f"{class_name}: "
            f"mean Gcnn = {class_g_cnn:.4f}, "
            f"mean Gtr = {class_g_tr:.4f}"
        )

In [8]:
# ============================================================
# CGLGCTNet computational-efficiency benchmark
# Run in a fresh runtime after defining the model classes.
# ============================================================

import copy
import gc
import os
import platform
import statistics
import subprocess
import sys
import tempfile
import threading
import time

import numpy as np
import torch
import torchvision

try:
    import psutil
except ImportError:
    raise ImportError(
        "Install psutil first with: pip install psutil"
    )


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

CHECKPOINT_PATH = "best_scg_lgt.pth"
NUM_CLASSES = 38
INPUT_SIZE = 224

CPU_THREADS = 1
WARMUP_ITERATIONS = 100
TIMED_ITERATIONS = 500


# ------------------------------------------------------------
# Load the exact trained architecture
# ------------------------------------------------------------

def load_CGLGCTNet(checkpoint_path):
    model = CGLGCTNet(num_classes=NUM_CLASSES)

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu"
    )

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint

    model.load_state_dict(state_dict, strict=True)
    model.eval()

    return model


model = load_CGLGCTNett(CHECKPOINT_PATH)


# ------------------------------------------------------------
# Parameters and buffers
# ------------------------------------------------------------

def get_parameter_statistics(model):
    total = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    non_trainable = sum(
        parameter.numel()
        for parameter in model.parameters()
        if not parameter.requires_grad
    )

    buffers = sum(
        buffer.numel()
        for buffer in model.buffers()
    )

    parameter_bytes = sum(
        parameter.numel() * parameter.element_size()
        for parameter in model.parameters()
    )

    buffer_bytes = sum(
        buffer.numel() * buffer.element_size()
        for buffer in model.buffers()
    )

    return {
        "total_parameters": total,
        "trainable_parameters": trainable,
        "non_trainable_parameters": non_trainable,
        "registered_buffers": buffers,
        "raw_state_size_mb": (
            parameter_bytes + buffer_bytes
        ) / 1_000_000,
        "raw_state_size_mib": (
            parameter_bytes + buffer_bytes
        ) / (1024 ** 2),
    }


# ------------------------------------------------------------
# Actual serialized weights-only size
# ------------------------------------------------------------

def get_serialized_size(model):
    with tempfile.NamedTemporaryFile(
        suffix=".pth",
        delete=False
    ) as temporary_file:
        temporary_path = temporary_file.name

    try:
        torch.save(
            model.state_dict(),
            temporary_path
        )

        size_bytes = os.path.getsize(temporary_path)

        return {
            "bytes": size_bytes,
            "mb": size_bytes / 1_000_000,
            "mib": size_bytes / (1024 ** 2),
        }

    finally:
        if os.path.exists(temporary_path):
            os.remove(temporary_path)


# ------------------------------------------------------------
# Theoretical MACs for this exact architecture
# ------------------------------------------------------------

def get_theoretical_macs(num_classes=38):
    # Shared stem
    stem_conv = 112 * 112 * 24 * 3 * 3 * 3
    stem_depthwise = 56 * 56 * 24 * 3 * 3
    stem_pointwise = 56 * 56 * 48 * 24

    shared_stem = (
        stem_conv
        + stem_depthwise
        + stem_pointwise
    )

    # First MBConv
    mbconv1_expand = 56 * 56 * 192 * 48
    mbconv1_depthwise = 28 * 28 * 192 * 3 * 3
    mbconv1_project = 28 * 28 * 64 * 192
    mbconv1_eca = 192 * 3

    # Second MBConv
    mbconv2_expand = 28 * 28 * 256 * 64
    mbconv2_depthwise = 14 * 14 * 256 * 3 * 3
    mbconv2_project = 14 * 14 * 96 * 256
    mbconv2_eca = 256 * 3

    final_eca = 96 * 3

    cnn_branch = (
        mbconv1_expand
        + mbconv1_depthwise
        + mbconv1_project
        + mbconv1_eca
        + mbconv2_expand
        + mbconv2_depthwise
        + mbconv2_project
        + mbconv2_eca
        + final_eca
    )

    # Transformer patch embedding
    num_tokens = 196
    embed_dim = 128
    feedforward_dim = 256
    transformer_depth = 2

    patch_embedding = (
        14 * 14 * embed_dim * 48 * 4 * 4
    )

    # Per Transformer encoder layer
    qkv_projection = (
        num_tokens * embed_dim * (3 * embed_dim)
    )

    attention_scores = (
        num_tokens * num_tokens * embed_dim
    )

    attention_value_product = (
        num_tokens * num_tokens * embed_dim
    )

    attention_output_projection = (
        num_tokens * embed_dim * embed_dim
    )

    feedforward_layers = (
        num_tokens * embed_dim * feedforward_dim
        + num_tokens * feedforward_dim * embed_dim
    )

    transformer_layer = (
        qkv_projection
        + attention_scores
        + attention_value_product
        + attention_output_projection
        + feedforward_layers
    )

    transformer_encoder = (
        transformer_depth * transformer_layer
    )

    transformer_branch = (
        patch_embedding + transformer_encoder
    )

    # Projection, fusion, and classification
    cnn_projection = 96 * 256
    transformer_projection = 128 * 256

    two_cross_gates = 2 * (
        256 * 64 + 64 * 256
    )

    classifier = 256 * num_classes

    output_modules = (
        cnn_projection
        + transformer_projection
        + two_cross_gates
        + classifier
    )

    total_macs = (
        shared_stem
        + cnn_branch
        + transformer_branch
        + output_modules
    )

    attention_matrix_macs = transformer_depth * (
        attention_scores
        + attention_value_product
    )

    return {
        "shared_stem_mmacs": shared_stem / 1e6,
        "cnn_branch_mmacs": cnn_branch / 1e6,
        "transformer_branch_mmacs": transformer_branch / 1e6,
        "attention_matrix_mmacs": attention_matrix_macs / 1e6,
        "output_modules_mmacs": output_modules / 1e6,
        "total_mmacs": total_macs / 1e6,
        # Convention: one MAC = two FLOPs
        "total_mflops": 2 * total_macs / 1e6,
        "total_gflops": 2 * total_macs / 1e9,
    }


# ------------------------------------------------------------
# Latency and batch-1 throughput
# ------------------------------------------------------------

def benchmark_latency(
    source_model,
    device,
    dtype=torch.float32,
    batch_size=1,
    warmup=100,
    iterations=500,
    cpu_threads=1,
):
    device = torch.device(device)

    if device.type == "cpu" and dtype != torch.float32:
        raise ValueError(
            "Use FP32 for the primary CPU benchmark."
        )

    if device.type == "cpu":
        torch.set_num_threads(cpu_threads)

    benchmark_model = copy.deepcopy(source_model)
    benchmark_model = benchmark_model.to(
        device=device,
        dtype=dtype
    )
    benchmark_model.eval()

    input_tensor = torch.randn(
        batch_size,
        3,
        INPUT_SIZE,
        INPUT_SIZE,
        device=device,
        dtype=dtype
    )

    timings_ms = []

    with torch.inference_mode():
        # Warm-up
        for _ in range(warmup):
            _ = benchmark_model(input_tensor)

        if device.type == "cuda":
            torch.cuda.synchronize(device)

            for _ in range(iterations):
                start_event = torch.cuda.Event(
                    enable_timing=True
                )
                end_event = torch.cuda.Event(
                    enable_timing=True
                )

                start_event.record()
                _ = benchmark_model(input_tensor)
                end_event.record()

                torch.cuda.synchronize(device)

                elapsed_ms = start_event.elapsed_time(
                    end_event
                )
                timings_ms.append(elapsed_ms)

        else:
            for _ in range(iterations):
                start_time = time.perf_counter_ns()

                _ = benchmark_model(input_tensor)

                end_time = time.perf_counter_ns()

                elapsed_ms = (
                    end_time - start_time
                ) / 1_000_000

                timings_ms.append(elapsed_ms)

    timings_ms = np.asarray(
        timings_ms,
        dtype=np.float64
    )

    median_ms = float(np.median(timings_ms))

    results = {
        "device": str(device),
        "precision": str(dtype),
        "batch_size": batch_size,
        "warmup_iterations": warmup,
        "timed_iterations": iterations,
        "mean_latency_ms": float(np.mean(timings_ms)),
        "std_latency_ms": float(np.std(timings_ms, ddof=1)),
        "median_latency_ms": median_ms,
        "p95_latency_ms": float(
            np.percentile(timings_ms, 95)
        ),
        "throughput_images_per_second": (
            batch_size * 1000.0 / median_ms
        ),
    }

    del benchmark_model
    del input_tensor

    if device.type == "cuda":
        torch.cuda.empty_cache()

    gc.collect()

    return results


# ------------------------------------------------------------
# GPU peak memory
# Includes model weights, input, outputs, and activations.
# ------------------------------------------------------------

def measure_gpu_peak_memory(
    source_model,
    dtype=torch.float32,
    batch_size=1,
):
    if not torch.cuda.is_available():
        return None

    gc.collect()
    torch.cuda.empty_cache()

    device = torch.device("cuda")

    benchmark_model = copy.deepcopy(source_model)
    benchmark_model = benchmark_model.to(
        device=device,
        dtype=dtype
    )
    benchmark_model.eval()

    input_tensor = torch.randn(
        batch_size,
        3,
        INPUT_SIZE,
        INPUT_SIZE,
        device=device,
        dtype=dtype
    )

    torch.cuda.reset_peak_memory_stats(device)

    with torch.inference_mode():
        _ = benchmark_model(input_tensor)

    torch.cuda.synchronize(device)

    results = {
        "peak_allocated_mb": (
            torch.cuda.max_memory_allocated(device)
            / 1_000_000
        ),
        "peak_allocated_mib": (
            torch.cuda.max_memory_allocated(device)
            / (1024 ** 2)
        ),
        "peak_reserved_mb": (
            torch.cuda.max_memory_reserved(device)
            / 1_000_000
        ),
        "peak_reserved_mib": (
            torch.cuda.max_memory_reserved(device)
            / (1024 ** 2)
        ),
    }

    del benchmark_model
    del input_tensor

    torch.cuda.empty_cache()
    gc.collect()

    return results


# ------------------------------------------------------------
# CPU peak process memory
# Run this in a fresh runtime for a defensible result.
# ------------------------------------------------------------

def measure_cpu_peak_rss(
    source_model,
    batch_size=1,
    iterations=100,
    sampling_interval=0.001,
    cpu_threads=1,
):
    torch.set_num_threads(cpu_threads)

    benchmark_model = copy.deepcopy(source_model)
    benchmark_model = benchmark_model.cpu().float().eval()

    input_tensor = torch.randn(
        batch_size,
        3,
        INPUT_SIZE,
        INPUT_SIZE,
        dtype=torch.float32
    )

    process = psutil.Process(os.getpid())

    gc.collect()

    baseline_rss = process.memory_info().rss
    peak_rss = [baseline_rss]
    stop_sampling = threading.Event()

    def sample_memory():
        while not stop_sampling.is_set():
            current_rss = process.memory_info().rss

            if current_rss > peak_rss[0]:
                peak_rss[0] = current_rss

            time.sleep(sampling_interval)

    sampler = threading.Thread(
        target=sample_memory,
        daemon=True
    )

    sampler.start()

    with torch.inference_mode():
        for _ in range(iterations):
            _ = benchmark_model(input_tensor)

    stop_sampling.set()
    sampler.join()

    results = {
        "baseline_process_rss_mb": baseline_rss / 1_000_000,
        "peak_process_rss_mb": peak_rss[0] / 1_000_000,
        "incremental_peak_mb": (
            peak_rss[0] - baseline_rss
        ) / 1_000_000,
    }

    del benchmark_model
    del input_tensor
    gc.collect()

    return results


# ------------------------------------------------------------
# Hardware and software environment
# ------------------------------------------------------------

def get_environment_information():
    information = {
        "operating_system": platform.platform(),
        "python_version": sys.version.replace("\n", " "),
        "pytorch_version": torch.__version__,
        "torchvision_version": torchvision.__version__,
        "cpu_name": platform.processor(),
        "physical_cpu_cores": psutil.cpu_count(
            logical=False
        ),
        "logical_cpu_cores": psutil.cpu_count(
            logical=True
        ),
        "system_ram_gb": (
            psutil.virtual_memory().total / 1e9
        ),
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
        "cpu_threads_used": CPU_THREADS,
    }

    if torch.cuda.is_available():
        properties = torch.cuda.get_device_properties(0)

        information.update({
            "gpu_name": torch.cuda.get_device_name(0),
            "gpu_memory_gb": (
                properties.total_memory / 1e9
            ),
            "gpu_compute_capability": (
                f"{properties.major}.{properties.minor}"
            ),
        })

        try:
            driver_version = subprocess.check_output(
                [
                    "nvidia-smi",
                    "--query-gpu=driver_version",
                    "--format=csv,noheader"
                ],
                text=True
            ).strip()

            information["gpu_driver_version"] = driver_version

        except Exception:
            information["gpu_driver_version"] = "Unavailable"

    return information


# ------------------------------------------------------------
# Run all measurements
# ------------------------------------------------------------

parameter_stats = get_parameter_statistics(model)
serialized_size = get_serialized_size(model)
complexity_stats = get_theoretical_macs(NUM_CLASSES)
environment = get_environment_information()

cpu_fp32_results = benchmark_latency(
    source_model=model,
    device="cpu",
    dtype=torch.float32,
    batch_size=1,
    warmup=WARMUP_ITERATIONS,
    iterations=TIMED_ITERATIONS,
    cpu_threads=CPU_THREADS,
)

cpu_memory_results = measure_cpu_peak_rss(
    source_model=model,
    batch_size=1,
    iterations=100,
    cpu_threads=CPU_THREADS,
)

gpu_fp32_results = None
gpu_fp16_results = None
gpu_fp32_memory = None
gpu_fp16_memory = None

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

    gpu_fp32_results = benchmark_latency(
        source_model=model,
        device="cuda",
        dtype=torch.float32,
        batch_size=1,
        warmup=WARMUP_ITERATIONS,
        iterations=TIMED_ITERATIONS,
    )

    gpu_fp32_memory = measure_gpu_peak_memory(
        source_model=model,
        dtype=torch.float32,
        batch_size=1,
    )

    gpu_fp16_results = benchmark_latency(
        source_model=model,
        device="cuda",
        dtype=torch.float16,
        batch_size=1,
        warmup=WARMUP_ITERATIONS,
        iterations=TIMED_ITERATIONS,
    )

    gpu_fp16_memory = measure_gpu_peak_memory(
        source_model=model,
        dtype=torch.float16,
        batch_size=1,
    )


print("\nPARAMETERS")
for key, value in parameter_stats.items():
    print(f"{key}: {value}")

print("\nSERIALIZED WEIGHTS-ONLY SIZE")
for key, value in serialized_size.items():
    print(f"{key}: {value}")

print("\nTHEORETICAL COMPLEXITY")
for key, value in complexity_stats.items():
    print(f"{key}: {value}")

print("\nCPU FP32 BATCH-1 PERFORMANCE")
for key, value in cpu_fp32_results.items():
    print(f"{key}: {value}")

print("\nCPU PEAK MEMORY")
for key, value in cpu_memory_results.items():
    print(f"{key}: {value}")

if gpu_fp32_results is not None:
    print("\nGPU FP32 BATCH-1 PERFORMANCE")
    for key, value in gpu_fp32_results.items():
        print(f"{key}: {value}")

    print("\nGPU FP32 PEAK MEMORY")
    for key, value in gpu_fp32_memory.items():
        print(f"{key}: {value}")

    print("\nGPU FP16 BATCH-1 PERFORMANCE")
    for key, value in gpu_fp16_results.items():
        print(f"{key}: {value}")

    print("\nGPU FP16 PEAK MEMORY")
    for key, value in gpu_fp16_memory.items():
        print(f"{key}: {value}")

print("\nENVIRONMENT")
for key, value in environment.items():
    print(f"{key}: {value}")


PARAMETERS
total_parameters: 595407
trainable_parameters: 595407
non_trainable_parameters: 0
registered_buffers: 2313
raw_state_size_mb: 2.390916
raw_state_size_mib: 2.2801551818847656

SERIALIZED WEIGHTS-ONLY SIZE
bytes: 2431151
mb: 2.431151
mib: 2.318526268005371

THEORETICAL COMPLEXITY
shared_stem_mmacs: 12.41856
cnn_branch_mmacs: 58.005088
transformer_branch_mmacs: 90.3168
attention_matrix_mmacs: 19.668992
output_modules_mmacs: 0.132608
total_mmacs: 160.873056
total_mflops: 321.746112
total_gflops: 0.321746112

CPU FP32 BATCH-1 PERFORMANCE
device: cpu
precision: torch.float32
batch_size: 1
warmup_iterations: 100
timed_iterations: 500
mean_latency_ms: 21.02293
std_latency_ms: 4.3788628544973855
median_latency_ms: 20.87795
p95_latency_ms: 26.515514999999994
throughput_images_per_second: 47.897422879161994

CPU PEAK MEMORY
baseline_process_rss_mb: 796.184576
peak_process_rss_mb: 796.205056
incremental_peak_mb: 0.02048

GPU FP32 BATCH-1 PERFORMANCE
device: cuda
precision: torch.float3